**1단계 파일 확인**

In [1]:
from google.colab import files

uploaded = files.upload()   # 지역별 선박입출항 실적 통계정보 22~24년도 불러옴 - 창원 항만권역의 물류 수요 증가 알아보기 위헤

Saving 지역별 선박입출항실적 통계정보_2022.csv to 지역별 선박입출항실적 통계정보_2022.csv
Saving 지역별 선박입출항실적 통계정보_2023.csv to 지역별 선박입출항실적 통계정보_2023.csv
Saving 지역별 선박입출항실적 통계정보_2024.csv to 지역별 선박입출항실적 통계정보_2024.csv


In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os

base_path = "/content/drive/MyDrive/contest_changwon_port"

folders = [
    "data/raw/port_calls",
    "data/raw/port_facility",
    "data/raw/ship_spec",
    "data/raw/trucking_company",
    "data/raw/freight_forwarder",
    "data/processed",
    "output/tables",
    "output/figures"
]

for folder in folders:
    path = os.path.join(base_path, folder)
    os.makedirs(path, exist_ok=True)
    print("생성/확인 완료:", path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/data/raw/port_calls
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/data/raw/port_facility
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/data/raw/ship_spec
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/data/raw/trucking_company
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/data/raw/freight_forwarder
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/data/processed
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/output/tables
생성/확인 완료: /content/drive/MyDrive/contest_changwon_port/output/figures


In [12]:
import os
import glob

base_path = "/content/drive/MyDrive/contest_changwon_port"
port_calls_path = os.path.join(base_path, "data/raw/port_calls")

print("폴더 존재:", os.path.exists(port_calls_path))

files = glob.glob(os.path.join(port_calls_path, "*"))

print("파일 개수:", len(files))

for f in files:
    print(os.path.basename(f))

폴더 존재: True
파일 개수: 3
지역별 선박입출항실적 통계정보_2024.csv
지역별 선박입출항실적 통계정보_2023.csv
지역별 선박입출항실적 통계정보_2022.csv


In [13]:
import pandas as pd
import os
import glob

base_path = "/content/drive/MyDrive/contest_changwon_port"
port_calls_path = os.path.join(base_path, "data/raw/port_calls")

files = glob.glob(os.path.join(port_calls_path, "*"))

target_years = ["2022", "2023", "2024"]

selected_files = [
    f for f in files
    if any(year in os.path.basename(f) for year in target_years)
]

print("선택된 파일 수:", len(selected_files))

for f in selected_files:
    print(os.path.basename(f))

선택된 파일 수: 3
지역별 선박입출항실적 통계정보_2024.csv
지역별 선박입출항실적 통계정보_2023.csv
지역별 선박입출항실적 통계정보_2022.csv


2단계 - 파일 불러와서 하나로 합치기

In [14]:
import pandas as pd
import os
import re

def read_file_korean(path):
    filename = os.path.basename(path)

    if filename.endswith(".csv"):
        for enc in ["cp949", "utf-8-sig", "utf-8", "euc-kr"]:
            try:
                df = pd.read_csv(path, encoding=enc)
                print(f"성공: {filename} / encoding={enc}")
                return df
            except UnicodeDecodeError:
                continue
        raise ValueError(f"인코딩 실패: {filename}")

    elif filename.endswith(".xlsx"):
        df = pd.read_excel(path)
        print(f"성공: {filename} / excel")
        return df

    else:
        raise ValueError(f"지원하지 않는 파일 형식: {filename}")


df_list = []

for file in selected_files:
    df = read_file_korean(file)

    # 파일명에서 연도 추출
    year_match = re.search(r"20\d{2}", os.path.basename(file))
    if year_match:
        df["year"] = int(year_match.group())

    df["source_file"] = os.path.basename(file)
    df_list.append(df)

port_calls = pd.concat(df_list, ignore_index=True)

print("통합 데이터 크기:", port_calls.shape)
display(port_calls.head())
print(port_calls.columns)

# 파일 불러와서 하나로 합치기



성공: 지역별 선박입출항실적 통계정보_2024.csv / encoding=utf-8-sig
성공: 지역별 선박입출항실적 통계정보_2023.csv / encoding=utf-8-sig
성공: 지역별 선박입출항실적 통계정보_2022.csv / encoding=utf-8-sig
통합 데이터 크기: (240, 9)


,useYm,prtAgNm,nm,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg,year,source_file
0,202401,서귀포,일본,0,0,2,272402.0,2024,지역별 선박입출항실적 통계정보_2024...
1,202401,서귀포,극동아시아 지역,0,0,4,443640.0,2024,지역별 선박입출항실적 통계정보_2024...
2,202401,서귀포,동남아시아 지역,0,0,0,0.0,2024,지역별 선박입출항실적 통계정보_2024...
3,202401,서귀포,서남아시아 지역,0,0,0,0.0,2024,지역별 선박입출항실적 통계정보_2024...
4,202401,서귀포,중동,0,0,0,0.0,2024,지역별 선박입출항실적 통계정보_2024...


Index(['useYm', 'prtAgNm', 'nm', 'intrvsslCo', 'intrvsslGrtg', 'fnshpCo',
       'fnshpGrtg', 'year', 'source_file'],
      dtype='object')


3단계 - 컬럼명 확인

In [15]:
for col in port_calls.columns:
    print(col)   # 컬럼명 확인

useYm
prtAgNm
nm
intrvsslCo
intrvsslGrtg
fnshpCo
fnshpGrtg
year
source_file


4단계 - 항만명 확인 // 5단계 - 창원 항만권역만 필터링

In [39]:
port_calls["prtAgNm"].unique() # 항만명들 알아보기

array(['서귀포', '울산', '옥포', '고현', '경인', '동해.묵호', '속초', '옥계', '호산', '대산',
       '군산', '장항', '완도', '진해', '제주'], dtype=object)

In [18]:
target_ports = ["마산", "진해", "신항", "부산신항"]   # 창원 항만권역 필터링

port_calls_changwon = port_calls[
    port_calls["prtAgNm"].astype(str).str.contains("|".join(target_ports), na=False)
].copy()

print("창원 항만권역 데이터 크기:", port_calls_changwon.shape)
display(port_calls_changwon.head())

창원 항만권역 데이터 크기: (12, 9)


,useYm,prtAgNm,nm,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg,year,source_file
168,202312,진해,일본,0,0,10,22463.0,2023,지역별 선박입출항실적 통계정보_2023...
169,202312,진해,극동아시아 지역,0,0,10,29901.0,2023,지역별 선박입출항실적 통계정보_2023...
170,202312,진해,동남아시아 지역,0,0,0,0.0,2023,지역별 선박입출항실적 통계정보_2023...
171,202312,진해,서남아시아 지역,0,0,2,7506.0,2023,지역별 선박입출항실적 통계정보_2023...
172,202312,진해,중동,0,0,0,0.0,2023,지역별 선박입출항실적 통계정보_2023...


부산신항은 창원 진해구와 연결된 신항 배후권역으로 볼 수 있지만, 보고서에선 마산항과 진해항 중심으로 분석하고, 신항은 창원 진해구와 연계된 배후권역으로 보조적 검토 실행함 이라 작성하는게 도움

6단계 - 숫자 컬럼 정리

In [20]:
num_cols = ["intrvsslCo", "intrvsslGrtg", "fnshpCo", "fnshpGrtg"] # 숫자컬럼 정리

In [21]:
def clean_number(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("-", "0", regex=False)
        .str.strip()
        .replace("nan", "0")
        .pipe(pd.to_numeric, errors="coerce")
        .fillna(0)
    )

num_cols = ["intrvsslCo", "intrvsslGrtg", "fnshpCo", "fnshpGrtg"]

for col in num_cols:
    if col in port_calls.columns:
        port_calls[col] = clean_number(port_calls[col])

7단계 - 분석용 합계 컬럼 만들기 -> 내국선박 + 외국선박 합쳐서 전체 선박척수와 전체 총톤수 만들기

In [22]:
port_calls["total_ship_count"] = port_calls["intrvsslCo"] + port_calls["fnshpCo"]   # 전체 선박척수 = 내국선박척수 + 외국선박척수

port_calls["total_gross_tonnage"] = port_calls["intrvsslGrtg"] + port_calls["fnshpGrtg"]   # 전체 총톤수 = 내국선박총톤수 + 외국선박총톤수

8단계 - useYm에서 연도와 월 추출

In [27]:
port_calls["useYm"] = port_calls["useYm"].astype(str)

port_calls["use_year"] = port_calls["useYm"].str[:4].astype(int)
port_calls["month"] = port_calls["useYm"].str[-2:].astype(int)

#  파일명 기준 연도랑 useYm 기준 연도가 다른지 확인하기 위해 useYm에서도 연도와 월을 뽑아보는 것

In [26]:
display(port_calls[["useYm", "year", "use_year", "month"]].head())

,useYm,year,use_year,month
0,202401,2024,2024,1
1,202401,2024,2024,1
2,202401,2024,2024,1
3,202401,2024,2024,1
4,202401,2024,2024,1


9단계 - 항만명 확인하기

In [28]:
print(port_calls["prtAgNm"].unique())

['서귀포' '울산' '옥포' '고현' '경인' '동해.묵호' '속초' '옥계' '호산' '대산' '군산' '장항' '완도' '진해'
 '제주']


In [29]:
target_ports = ["진해"]

port_calls_jinhae = port_calls[
    port_calls["prtAgNm"].astype(str).str.contains("|".join(target_ports), na=False)
].copy()

print("진해항 데이터 크기:", port_calls_jinhae.shape)
display(port_calls_jinhae.head())

진해항 데이터 크기: (12, 13)


,useYm,prtAgNm,nm,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg,year,source_file,total_ship_count,total_gross_tonnage,use_year,month
168,202312,진해,일본,0,0,10,22463.0,2023,지역별 선박입출항실적 통계정보_2023...,10,22463.0,2023,12
169,202312,진해,극동아시아 지역,0,0,10,29901.0,2023,지역별 선박입출항실적 통계정보_2023...,10,29901.0,2023,12
170,202312,진해,동남아시아 지역,0,0,0,0.0,2023,지역별 선박입출항실적 통계정보_2023...,0,0.0,2023,12
171,202312,진해,서남아시아 지역,0,0,2,7506.0,2023,지역별 선박입출항실적 통계정보_2023...,2,7506.0,2023,12
172,202312,진해,중동,0,0,0,0.0,2023,지역별 선박입출항실적 통계정보_2023...,0,0.0,2023,12


10단계 - 진해항 데이터 확인 // 진해항만 잘 걸러졌는지 확인

In [30]:
print(port_calls_jinhae["prtAgNm"].unique())
print(port_calls_jinhae["year"].unique())
print(port_calls_jinhae[["useYm", "prtAgNm", "nm", "intrvsslCo", "intrvsslGrtg", "fnshpCo", "fnshpGrtg"]].head())

['진해']
[2023]
      useYm prtAgNm        nm  intrvsslCo  intrvsslGrtg  fnshpCo  fnshpGrtg
168  202312      진해        일본           0             0       10    22463.0
169  202312      진해  극동아시아 지역           0             0       10    29901.0
170  202312      진해  동남아시아 지역           0             0        0        0.0
171  202312      진해  서남아시아 지역           0             0        2     7506.0
172  202312      진해        중동           0             0        0        0.0


11단계 - 연도별 집계표 만들기 // 진해항의 연도별 내국선박, 외국선박, 전체 선박척수, 전체 총톤수를 합산

In [31]:
port_calls_yearly = (
    port_calls_jinhae
    .groupby("year", as_index=False)
    .agg({
        "intrvsslCo": "sum",
        "intrvsslGrtg": "sum",
        "fnshpCo": "sum",
        "fnshpGrtg": "sum",
        "total_ship_count": "sum",
        "total_gross_tonnage": "sum"
    })
)

display(port_calls_yearly)

,year,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg,total_ship_count,total_gross_tonnage
0,2023,0,0,22,59870.0,22,59870.0


In [32]:
print(port_calls["year"].value_counts().sort_index())

year
2022     24
2023    192
2024     24
Name: count, dtype: int64


In [33]:
port_calls[port_calls["prtAgNm"].astype(str).str.contains("진해", na=False)]["year"].value_counts().sort_index()ㅎ확인

,count
year,
2023,12


22-24에 진해 데이터는 23년도에 12개 뿐, 비교 불가능으로 2023년 진해항 수요 현황 보조 자료로 사용하기

In [34]:
# 진해항 데이터만 필터링
port_calls_jinhae = port_calls[
    port_calls["prtAgNm"].astype(str).str.contains("진해", na=False)
].copy()

print(port_calls_jinhae.shape)
display(port_calls_jinhae.head())

# 연도별 증가율 계산 뺴고, 2023년 월별 현황 정리

(12, 13)


,useYm,prtAgNm,nm,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg,year,source_file,total_ship_count,total_gross_tonnage,use_year,month
168,202312,진해,일본,0,0,10,22463.0,2023,지역별 선박입출항실적 통계정보_2023...,10,22463.0,2023,12
169,202312,진해,극동아시아 지역,0,0,10,29901.0,2023,지역별 선박입출항실적 통계정보_2023...,10,29901.0,2023,12
170,202312,진해,동남아시아 지역,0,0,0,0.0,2023,지역별 선박입출항실적 통계정보_2023...,0,0.0,2023,12
171,202312,진해,서남아시아 지역,0,0,2,7506.0,2023,지역별 선박입출항실적 통계정보_2023...,2,7506.0,2023,12
172,202312,진해,중동,0,0,0,0.0,2023,지역별 선박입출항실적 통계정보_2023...,0,0.0,2023,12


In [35]:
# 전체 선박척수, 전체 총톤수 만들기

num_cols = ["intrvsslCo", "intrvsslGrtg", "fnshpCo", "fnshpGrtg"]

for col in num_cols:
    port_calls_jinhae[col] = pd.to_numeric(port_calls_jinhae[col], errors="coerce").fillna(0)

port_calls_jinhae["total_ship_count"] = (
    port_calls_jinhae["intrvsslCo"] + port_calls_jinhae["fnshpCo"]
)

port_calls_jinhae["total_gross_tonnage"] = (
    port_calls_jinhae["intrvsslGrtg"] + port_calls_jinhae["fnshpGrtg"]
)

In [36]:
# 2023년 월별 진해항 수요 현황 만들기

port_calls_jinhae["useYm"] = port_calls_jinhae["useYm"].astype(str)
port_calls_jinhae["date"] = pd.to_datetime(port_calls_jinhae["useYm"], format="%Y%m")

jinhae_monthly_2023 = (
    port_calls_jinhae
    .groupby(["useYm", "date"], as_index=False)
    .agg({
        "intrvsslCo": "sum",
        "intrvsslGrtg": "sum",
        "fnshpCo": "sum",
        "fnshpGrtg": "sum",
        "total_ship_count": "sum",
        "total_gross_tonnage": "sum"
    })
)

display(jinhae_monthly_2023)

,useYm,date,intrvsslCo,intrvsslGrtg,fnshpCo,fnshpGrtg,total_ship_count,total_gross_tonnage
0,202312,2023-12-01,0,0,22,59870.0,22,59870.0


In [37]:
# 2023년 진해항 총합 만들기

jinhae_summary_2023 = pd.DataFrame({
    "year": [2023],
    "domestic_ship_count": [port_calls_jinhae["intrvsslCo"].sum()],
    "domestic_gross_tonnage": [port_calls_jinhae["intrvsslGrtg"].sum()],
    "foreign_ship_count": [port_calls_jinhae["fnshpCo"].sum()],
    "foreign_gross_tonnage": [port_calls_jinhae["fnshpGrtg"].sum()],
    "total_ship_count": [port_calls_jinhae["total_ship_count"].sum()],
    "total_gross_tonnage": [port_calls_jinhae["total_gross_tonnage"].sum()]
})

display(jinhae_summary_2023)

,year,domestic_ship_count,domestic_gross_tonnage,foreign_ship_count,foreign_gross_tonnage,total_ship_count,total_gross_tonnage
0,2023,0,0,22,59870.0,22,59870.0


해석= 2023년 진해항 입출항실적은 외국선박 중심으로 나타났으며 전체 선박척수는 22척, 전체 총톤수는 59870톤으로 집계되었다

저장 코드

In [38]:
processed_path = os.path.join(base_path, "data/processed")
os.makedirs(processed_path, exist_ok=True)

jinhae_monthly_2023.to_csv(
    os.path.join(processed_path, "jinhae_port_calls_monthly_2023.csv"),
    index=False,
    encoding="utf-8-sig"
)

jinhae_summary_2023.to_csv(
    os.path.join(processed_path, "jinhae_port_calls_summary_2023.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("진해항 2023 입출항실적 저장 완료")

진해항 2023 입출항실적 저장 완료
